# Day 05 · AI 輔助開發：Code with AI

> 第一部・新兵入伍　|　🛠️ 工具設定為主

**前置需求**：（無特殊需求，不需要 API 金鑰）

**對應文章**：`Day 05 - AI 輔助開發：Code with AI.md`

## 今天要學會

1. 把 ADK 文件接進你的 coding agent（MCP / skills / `llms.txt` 三條路）
2. 評估哪一條路適合你的工作流

> 這天沒有 agent 程式碼，是**工具設定**。原文給了指令但沒有驗證方式，
> 本日補上可執行的檢查——設定完到底有沒有生效，跑一下就知道。

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. 為什麼需要這一天

ADK 更新很快。你的 coding agent（Claude Code、Cursor、Copilot…）的訓練資料
大概率停在 1.x，會很有自信地寫出 `from google.adk import ...` 之類**已經
不存在**的 API。

三條路可以把最新文件餵給它：

| 路線 | 機制 | 需要網路 | 適合 |
|---|---|---|---|
| **A. `llms.txt`** | 一份純文字索引，直接貼給模型 | 抓一次就好 | 最簡單、離線可用 |
| **B. Docs MCP Server** | 讓 agent 自己去查文件 | 每次查詢 | 涵蓋率最好 |
| **C. ADK skills** | 把官方 skill 包裝進 coding agent | 安裝一次 | 深度整合 |

## 2. 路線 A：`llms.txt`

`llms.txt` 是給 LLM 讀的網站索引，已經是滿普遍的慣例。ADK 的在
<https://adk.dev/llms.txt>。

In [2]:
import urllib.error
import urllib.request

URL = "https://adk.dev/llms.txt"

try:
    with urllib.request.urlopen(URL, timeout=20) as resp:
        content = resp.read().decode("utf-8", errors="replace")
    print(f"✅ 抓到 {len(content):,} 字元 / {len(content.splitlines()):,} 行\n")
    print("--- 前 25 行 ---")
    for line in content.splitlines()[:25]:
        print(" ", line)
except Exception as exc:
    content = ""
    print(f"⚠️ 抓不到（{type(exc).__name__}）：{exc}")
    print("可能是沒有網路。這一節可以跳過，不影響其他天。")

✅ 抓到 17,242 字元 / 252 行

--- 前 25 行 ---
  # Agent Development Kit (ADK)
  
  > Build powerful multi-agent systems with Agent Development Kit (ADK)
  
  An open-source, code-first toolkit for building, evaluating, and deploying sophisticated AI agents with flexibility and control.
  
  ## Build Agents
  
  - [Get started](https://adk.dev/get-started/index.md)
  - [Technical Overview](https://adk.dev/get-started/about/index.md)
  - [Agents CLI](https://adk.dev/get-started/agents-cli/index.md)
  - [Go](https://adk.dev/get-started/go/index.md)
  - [Google Cloud](https://adk.dev/get-started/google-cloud/index.md)
  - [Installation](https://adk.dev/get-started/installation/index.md)
  - [Java](https://adk.dev/get-started/java/index.md)
  - [Kotlin](https://adk.dev/get-started/kotlin/index.md)
  - [Python](https://adk.dev/get-started/python/index.md)
  - [TypeScript](https://adk.dev/get-started/typescript/index.md)
  - [Build your agent with ADK](https://adk.dev/tutorials/index.md)
  - [Agent 

### 它涵蓋了什麼

In [3]:
import re

if content:
    links = re.findall(r"\[([^\]]+)\]\(([^)]+)\)", content)
    print(f"索引裡有 {len(links)} 個連結\n")
    topics = ["workflow", "tool", "session", "plugin", "a2a", "eval",
              "deploy", "skill", "memory", "artifact", "callback", "live"]
    print("30 天會用到的主題，涵蓋狀況：")
    low = content.lower()
    for t in topics:
        n = low.count(t)
        print(f"  {'✅' if n else '❌'} {t:10s} 出現 {n:3d} 次")

索引裡有 231 個連結

30 天會用到的主題，涵蓋狀況：
  ✅ workflow   出現  20 次
  ✅ tool       出現  78 次
  ✅ session    出現  16 次
  ✅ plugin     出現  14 次
  ✅ a2a        出現  14 次
  ✅ eval       出現  10 次
  ✅ deploy     出現  13 次
  ✅ skill      出現   4 次
  ✅ memory     出現   5 次
  ✅ artifact   出現   2 次
  ✅ callback   出現   7 次
  ✅ live       出現  15 次


### 怎麼用

最簡單：把整份存下來，需要時貼給你的 coding agent。

In [4]:
from pathlib import Path

if content:
    out = Path.cwd() / "adk_llms.txt"
    out.write_text(content, encoding="utf-8")
    print(f"已存到 {out.name}（{out.stat().st_size:,} bytes）")
    print("\n用法：把這個檔案拖進對話，或在 prompt 開頭說")
    print("『以下是 Google ADK 的最新文件索引，請以此為準：...』")

已存到 adk_llms.txt（17,244 bytes）

用法：把這個檔案拖進對話，或在 prompt 開頭說
『以下是 Google ADK 的最新文件索引，請以此為準：...』


> **注意**：`llms.txt` 是**索引**不是全文。它列出有哪些頁面，
> 模型還是得去讀個別頁面才知道細節。這是它跟 MCP 的主要差別。

## 3. 路線 B：ADK Docs MCP Server

讓你的 coding agent 能**主動查**文件，而不是靠你貼。

### Claude Code

```bash
claude mcp add adk-docs --transport stdio -- \
  uvx --from mcpdoc mcpdoc --urls AgentDevelopmentKit:https://adk.dev/llms.txt
```

### 其他支援 MCP 的工具（Cursor / VS Code / Windsurf…）

寫進設定檔的 `mcpServers` 區塊：

In [5]:
import json

mcp_config = {
    "mcpServers": {
        "adk-docs": {
            "command": "uvx",
            "args": [
                "--from", "mcpdoc", "mcpdoc",
                "--urls", "AgentDevelopmentKit:https://adk.dev/llms.txt",
            ],
        }
    }
}
print(json.dumps(mcp_config, indent=2))

{
  "mcpServers": {
    "adk-docs": {
      "command": "uvx",
      "args": [
        "--from",
        "mcpdoc",
        "mcpdoc",
        "--urls",
        "AgentDevelopmentKit:https://adk.dev/llms.txt"
      ]
    }
  }
}


### 📌 補充：先確認 `uvx` 真的在

原文直接給指令，但如果 `uv` 沒裝，上面兩種設定都會**靜默失敗**——
MCP server 起不來，你的 agent 只會覺得「沒有這個工具」，不會報錯。

In [6]:
import shutil
import subprocess

uvx = shutil.which("uvx")
uv = shutil.which("uv")
print(f"uv  : {uv or '❌ 找不到'}")
print(f"uvx : {uvx or '❌ 找不到'}")

if uvx:
    r = subprocess.run([uvx, "--version"], capture_output=True, text=True, timeout=60)
    print(f"版本: {r.stdout.strip() or r.stderr.strip()}")
else:
    print("\n安裝：curl -LsSf https://astral.sh/uv/install.sh | sh")

uv  : /root/.local/bin/uv
uvx : /root/.local/bin/uvx
版本: uvx 0.12.5 (aarch64-unknown-linux-gnu)


### 驗證設定 JSON 的正確性

MCP 設定寫錯是最常見的問題，而且通常不會有明顯錯誤訊息。

In [7]:
def check_mcp_config(cfg: dict) -> list[str]:
    """檢查 MCP 設定的常見錯誤。"""
    problems = []
    servers = cfg.get("mcpServers")
    if not isinstance(servers, dict) or not servers:
        return ["缺少 mcpServers，或它不是物件"]
    for name, spec in servers.items():
        if "command" not in spec:
            problems.append(f"{name}: 缺少 command")
        elif not shutil.which(spec["command"]):
            problems.append(f"{name}: command '{spec['command']}' 在 PATH 裡找不到")
        args = spec.get("args", [])
        if not isinstance(args, list):
            problems.append(f"{name}: args 必須是陣列")
        elif any(not isinstance(a, str) for a in args):
            problems.append(f"{name}: args 裡有非字串元素")
    return problems


issues = check_mcp_config(mcp_config)
if issues:
    print("⚠️ 發現問題:")
    for i in issues:
        print("  -", i)
else:
    print("✅ 設定看起來沒問題")

# 反例
bad = {"mcpServers": {"broken": {"args": "--urls x", "command": "definitely-not-installed"}}}
print("\n反例檢查:")
for i in check_mcp_config(bad):
    print("  -", i)

✅ 設定看起來沒問題

反例檢查:
  - broken: command 'definitely-not-installed' 在 PATH 裡找不到
  - broken: args 必須是陣列


## 4. 路線 C：ADK Skills

```bash
uvx google-agents-cli setup
```

這會把官方維護的 ADK skill 包裝安裝進你的 coding agent，
讓它在寫 ADK 程式碼時自動載入正確的 API 慣例。

Skill 本身是什麼、怎麼自己做一個，是 **Day 25** 的主題。

In [8]:
# 看看這台機器上有沒有已經裝好的 skill
candidates = [
    Path.home() / ".claude" / "skills",
    Path.cwd().parents[1] / ".claude" / "skills",
]
found = False
for d in candidates:
    if d.exists():
        skills = sorted(p.name for p in d.iterdir() if p.is_dir())
        print(f"{d}: {len(skills)} 個 skill")
        for s in skills[:10]:
            mark = "⭐" if "adk" in s.lower() or "agent" in s.lower() else "  "
            print(f"  {mark} {s}")
        found = True
if not found:
    print("這台機器上沒有找到 skills 目錄（正常，代表還沒安裝）")

/root/.claude/skills: 7 個 skill
  ⭐ google-agents-cli-adk-code
  ⭐ google-agents-cli-deploy
  ⭐ google-agents-cli-eval
  ⭐ google-agents-cli-observability
  ⭐ google-agents-cli-publish
  ⭐ google-agents-cli-scaffold
  ⭐ google-agents-cli-workflow


## 5. 📌 補充：三條路怎麼選

原文列了三條但沒說怎麼挑。實際差異：

| | llms.txt | MCP Server | Skills |
|---|---|---|---|
| 設定成本 | 最低（抓一個檔） | 中（改設定檔） | 中（跑一個指令） |
| 每次查詢延遲 | 無（已在 context 裡） | 有（要連網） | 無 |
| 涵蓋深度 | 只有索引 | **可讀完整頁面** | 精選重點 |
| 佔用 context | 大（整份貼進去） | 小（按需查） | 小 |
| 離線可用 | ✅ | ❌ | ✅ |
| 內容新鮮度 | 你抓的那一刻 | **即時** | 隨 skill 更新 |

**建議**：

- 只是偶爾寫 ADK → **llms.txt**，最省事
- 天天寫、需要查細節 → **MCP Server**，涵蓋率最好
- 想要 agent 自動遵守慣例 → **Skills**
- 三者不衝突，可以都裝

## 6. 📌 補充：怎麼驗證真的有效

設定完之後，用一個**只有新版才答得出來**的問題去測你的 coding agent。
下面這幾題的正確答案，這個 repo 的前四天都驗證過：

In [9]:
QUIZ = [
    ("ADK 2.x 裡，條件分支要怎麼寫？",
     "用 Workflow + ctx.route，邊寫成 dict。答『用 SequentialAgent』就是舊資料。"),
    ("SqliteSessionService 的建構參數叫什麼？",
     "db_path（不是 db_url，那是 DatabaseSessionService 的）。"),
    ("Plugin 的 callback 跟 agent 的 callback 誰先執行？",
     "Plugin 先，而且 Plugin 短路時 agent 的完全不會被呼叫。"),
    ("output_schema 可以跟 tools 並存嗎？",
     "ADK 2.8 可以。答『絕對不行』是舊版行為。"),
    ("Workflow 節點的輸出放在 Event 的哪個欄位？",
     "event.output，不是 event.content。"),
]

for i, (q, a) in enumerate(QUIZ, 1):
    print(f"{i}. {q}")
    print(f"   ✅ {a}\n")

1. ADK 2.x 裡，條件分支要怎麼寫？
   ✅ 用 Workflow + ctx.route，邊寫成 dict。答『用 SequentialAgent』就是舊資料。

2. SqliteSessionService 的建構參數叫什麼？
   ✅ db_path（不是 db_url，那是 DatabaseSessionService 的）。

3. Plugin 的 callback 跟 agent 的 callback 誰先執行？
   ✅ Plugin 先，而且 Plugin 短路時 agent 的完全不會被呼叫。

4. output_schema 可以跟 tools 並存嗎？
   ✅ ADK 2.8 可以。答『絕對不行』是舊版行為。

5. Workflow 節點的輸出放在 Event 的哪個欄位？
   ✅ event.output，不是 event.content。



拿這五題去問你剛設定好的 coding agent。答錯代表它還在用舊資料，
設定沒生效。

## 7. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| MCP 設定好了但 agent 說沒有工具 | `uvx` 不在 PATH。MCP server 起不來是**靜默失敗** |
| agent 還是寫出 1.x 的 API | 文件源沒生效，或它優先信任自己的訓練資料——用第 6 節的題目驗證 |
| `llms.txt` 貼了還是答錯細節 | 它只是**索引**不是全文，細節要另外查 |
| 設定檔改了沒反應 | 多數工具要**重啟**才會重新載入 MCP server |

## 8. 動手練習

1. 把第 2 節存下來的 `adk_llms.txt` 貼給你的 coding agent，
   問第 6 節的五個問題，記錄答對幾題。
2. 裝好 MCP server 之後再問一次同樣五題，比較差異。
3. 用 `check_mcp_config()` 檢查你自己真正的 MCP 設定檔。

## 本日回顧

- **你的 coding agent 大概率停在 ADK 1.x**，會很有自信地寫出已不存在的 API。
- **三條路**：`llms.txt`（最省事、離線可用、但只有索引）、
  MCP Server（涵蓋最好、需連網）、Skills（自動遵守慣例）。三者可並存。
- **MCP 設定失敗是靜默的**——先確認 `uvx` 在 PATH 裡。
- **一定要驗證**：用只有新版才答得出來的問題去測，別假設設定就等於生效。

---
**下一天 → `../day06_custom_tools/`**